# Laboratorio 4 - RAG y evaluacion

**Duracion estimada:** 45 minutos

**Objetivo**

- Montar un flujo RAG de extremo a extremo.
- Forzar una politica clara de abstencion.
- Evaluar respuestas con preguntas respondibles y no respondibles.

**Prerequisitos**

- Stack Docker levantado
- Modelos disponibles en Ollama
- Coleccion indexada o capacidad de rehacerla

**Criterios de exito**

- Defines `retrieve`, `build_context` y `ask_rag`.
- Respondes una pregunta presente en los documentos.
- Te abstienes razonablemente en una pregunta fuera de los documentos.


## Antes de empezar

Si no estas seguro de que la coleccion existe, puedes rehacer la indexacion desde el laboratorio 3.
En la solucion de este notebook se incluye una ayuda para reconstruirla si hace falta.


In [ ]:
import os
import uuid
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, PointStruct, VectorParams

load_dotenv(Path("..").resolve() / ".env")

OLLAMA_PORT = os.getenv("OLLAMA_PORT", "11434")
QDRANT_PORT = os.getenv("QDRANT_PORT", "6333")
CHAT_MODEL = os.getenv("CHAT_MODEL", "llama3")
EMBEDDING_MODEL = os.getenv("EMBEDDING_MODEL", "embeddinggemma")
COLLECTION_NAME = os.getenv("COLLECTION_NAME", "frasohome_docs")
CHUNK_SIZE = int(os.getenv("CHUNK_SIZE", "650"))
CHUNK_OVERLAP = int(os.getenv("CHUNK_OVERLAP", "100"))
TOP_K = int(os.getenv("TOP_K", "3"))

OLLAMA_BASE_URL = f"http://127.0.0.1:{OLLAMA_PORT}"
QDRANT_URL = f"http://127.0.0.1:{QDRANT_PORT}"
DOCS_DIR = Path("..").resolve() / "docs"
EVAL_PATH = Path("..").resolve() / "evaluation" / "questions.csv"

chat_model = ChatOllama(model=CHAT_MODEL, base_url=OLLAMA_BASE_URL, temperature=0)
embedding_model = OllamaEmbeddings(model=EMBEDDING_MODEL, base_url=OLLAMA_BASE_URL)
client = QdrantClient(url=QDRANT_URL)

print(
    {
        "COLLECTION_NAME": COLLECTION_NAME,
        "TOP_K": TOP_K,
        "EVAL_PATH": str(EVAL_PATH),
    }
)


## Paso 1 - Definir un prompt con abstencion

Construye un prompt que obligue al modelo a:

- usar solo el contexto recuperado;
- decir claramente que no lo sabe si falta informacion.


In [ ]:
# TODO: define `prompt`, `chain` y usa `StrOutputParser`.
prompt = None
chain = None

assert prompt is not None
assert chain is not None


## Paso 2 - Implementar retrieval y ensamblado de contexto

Completa estas tres funciones:

- `retrieve(question, top_k=TOP_K)`
- `build_context(hits)`
- `ask_rag(question, top_k=TOP_K)`


In [ ]:
def retrieve(question: str, top_k: int = TOP_K):
    raise NotImplementedError("Completa el retrieval")


def build_context(hits):
    raise NotImplementedError("Construye el bloque de contexto")


def ask_rag(question: str, top_k: int = TOP_K):
    raise NotImplementedError("Une retrieval + prompt + generacion")


### Checkpoint 1

Antes de seguir, deberias tener claro:

- donde se genera el embedding de consulta;
- donde se construye el contexto textual;
- donde se aplica la instruccion de abstencion.


## Paso 3 - Ejecutar una pregunta respondible y una no respondible

Usa estas dos preguntas:

- `Puedo devolver una mesa ya montada si simplemente no me convence?`
- `FrasoHome ofrece servicio de montaje a domicilio?`


In [ ]:
answerable_question = "Puedo devolver una mesa ya montada si simplemente no me convence?"
unanswerable_question = "FrasoHome ofrece servicio de montaje a domicilio?"

# TODO: ejecuta `ask_rag` para ambas y muestra respuesta y fuentes.
raise NotImplementedError("Completa las pruebas manuales")


## Paso 4 - Evaluar un lote de preguntas

Carga `evaluation/questions.csv`, ejecuta al menos las 5 primeras preguntas y crea un DataFrame con:

- `question`
- `expected_source`
- `top_source`
- `should_abstain`
- `answer`


In [ ]:
df = pd.read_csv(EVAL_PATH).head(5).copy()

# TODO: recorre el DataFrame y construye `results`.
results = []

pd.DataFrame(results)


## Paso 5 - Comparar una configuracion

Repite una pregunta con `top_k=TOP_K` y luego con `top_k=5`. Observa si cambia:

- la calidad del contexto,
- la respuesta final,
- el riesgo de meter ruido.


In [ ]:
comparison_question = "Que herramientas necesito para montar la mesa Oslo?"

# TODO: compara `ask_rag(comparison_question, top_k=TOP_K)` con `ask_rag(..., top_k=5)`.
raise NotImplementedError("Completa la comparacion de configuracion")


### Checkpoint 2

Verifica:

- tienes al menos una respuesta sustentada por el contexto;
- tienes al menos un caso donde el sistema debe abstenerse;
- ya puedes razonar si `TOP_K` mejora recall o introduce ruido.


## Reflexion 1

Responde brevemente:

1. En que se diferencia "recuperar bien" de "responder bien"?
2. Por que una mala recuperacion puede contaminar una respuesta aunque el modelo sea bueno?


## Reflexion 2

Responde brevemente:

1. Cuando deberia abstenerse un sistema RAG?
2. Si tuvieras que priorizar una mejora, cual tocarias primero: prompt, chunking, corpus o retrieval?

**Mini extension opcional**

Repite la evaluacion despues de reindexar con otra configuracion de chunking y compara resultados.
